In [1]:
import sys
import os

# Set project root (adjust if needed)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import sys
print(sys.executable)


c:\Users\shaha\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
import psycopg2
print(psycopg2.__version__)

2.9.11 (dt dec pq3 ext lo64)


In [7]:
!pip install psycopg2
!pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --------------------------------- ------ 2.4/2.8 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 8.0 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [4]:
from src.utils.query_DB import query_database, parse_recipes_query_result, parse_usda_foods_query_result, TABLE_NAMES


In [5]:
all_recipes_query = "SELECT * FROM recipes;"

message, df = query_database(all_recipes_query)
print(message)
df.count()

Error querying Supabase: (psycopg2.errors.QueryCanceled) canceling statement due to statement timeout

[SQL: SELECT * FROM recipes;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


AttributeError: 'NoneType' object has no attribute 'count'

In [14]:
import pandas as pd
recipes_csv_path = 'C:\\Users\\shaha\\Documents\\technion\\year 4 sem 1\\chosen_subjects\\Nutrissistant\\data\\recipes.csv'
# recepies_df = pd.read_csv('C:\\Users\\shaha\\Documents\\technion\\year 4 sem 1\\chosen_subjects\\Nutrissistant\\data\\recipes.csv')


In [20]:
recepies_df.columns

Index(['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime',
       'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory',
       'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts',
       'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent',
       'SaturatedFatContent', 'CholesterolContent', 'SodiumContent',
       'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent',
       'RecipeServings', 'RecipeYield', 'RecipeInstructions'],
      dtype='object')

In [21]:
def clean_r_vector_string(text):
    """
    Cleans R-style string vectors like: c(""sugar"", ""lemons"")
    Returns clean text: sugar, lemons
    """
    if pd.isna(text):
        return text
        
    text = str(text).strip()
    # Remove the surrounding c(...)
    if text.startswith('c(') and text.endswith(')'):
        text = text[2:-1]
        
    # Remove all the double and single quotes
    text = text.replace('"', '').replace("'", "")
    return text.strip()

def filter_recipe_database(path):
    # Load the raw dataset
    print("Loading recipes.csv...")
    
    # Use the exact column names from CSV
    target_columns = [
        'Name', 'TotalTime', 'Keywords', 
        'RecipeIngredientParts', 'RecipeInstructions', 'Calories', 'RecipeCategory', 
        'FatContent', 'SaturatedFatContent', 'CholesterolContent','CarbohydrateContent', 'SugarContent', 'FiberContent', 'ProteinContent'
    ]
    
    df = pd.read_csv(path, usecols=target_columns)

    # Rename columns to be lowercased and SQL-friendly
    df = df.rename(columns={
        'Name': 'name',
        'TotalTime': 'total_time',
        'Keywords': 'tags',
        'RecipeCategory': 'category',
        'RecipeIngredientParts': 'ingredients',
        'RecipeInstructions': 'steps',
        'Calories': 'calories'
    })

    # Drop rows with missing critical data
    df = df.dropna(subset=['name', 'total_time', 'ingredients', 'steps'])

    print("Converting ISO 8601 durations to integer minutes...")
    
    # Extract hours and minutes into separate columns using regex
    # It looks for "PT", then optionally extracts digits before "H", and optionally digits before "M"
    extracted_time = df['total_time'].str.extract(r'PT(?:(?P<hours>\d+)H)?(?:(?P<minutes>\d+)M)?')
    
    # Fill missing values with 0 and convert to floats for calculation
    extracted_time = extracted_time.fillna(0).astype(float)
    
    # Calculate total minutes
    df['minutes'] = (extracted_time['hours'] * 60) + extracted_time['minutes']
    
    # Convert back to an integer
    df['minutes'] = df['minutes'].astype(int)
    
    # Drop the old 'total_time' column
    df = df.drop(columns=['total_time'])

    # Filter out outliers (keeping recipes under 3 hours / 180 mins)
    df = df[df['minutes'] <= 180]
    df = df[df['minutes'] > 0]

    # Clean the R-style vector strings
    print("Cleaning ingredient and instruction formatting...")
    cols_to_clean = ['tags', 'ingredients', 'steps']
    for col in cols_to_clean:
        df[col] = df[col].apply(clean_r_vector_string)

    print(f"Filtered down to {len(df)} highly usable recipes.")
    return df

In [22]:
recepies_df_filterd = filter_recipe_database(recipes_csv_path)
recepies_df_filterd.head()

Loading recipes.csv...
Converting ISO 8601 durations to integer minutes...
Cleaning ingredient and instruction formatting...
Filtered down to 480535 highly usable recipes.


,name,category,tags,ingredients,calories,FatContent,SaturatedFatContent,CholesterolContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,steps,minutes
2,Best Lemonade,Beverages,"Low Protein, Low Cholesterol, Healthy, Summer,...","sugar, lemons, rind of, lemon, zest of, fresh ...",311.1,0.2,0.0,0.0,81.5,0.4,77.2,0.3,"Into a 1 quart Jar with tight fitting lid, put...",35
4,Cabbage Soup,Vegetable,"Low Protein, Vegan, Low Cholesterol, Healthy, ...","plain tomato juice, cabbage, onion, carrots, c...",103.6,0.4,0.1,0.0,25.1,4.8,17.7,4.3,"Mix everything together and bring to a boil., ...",50
5,Best Blackbottom Pie,Pie,"Dessert, Weeknight, Stove Top, < 4 Hours","graham cracker crumbs, sugar, butter, sugar, c...",437.9,19.3,10.9,94.3,58.0,1.8,42.5,7.0,"Graham Cracker Crust: In small bowl, combine g...",140
6,Warm Chicken A La King,Chicken,"Poultry, Meat, < 60 Mins","chicken, butter, flour, milk, celery, button m...",895.5,66.8,31.9,405.8,29.1,3.1,5.0,45.3,"Melt 1 1/2 ozs butter, add the flour and cook ...",38
7,Buttermilk Pie With Gingersnap Crumb Crust,Pie,"Dessert, Healthy, Weeknight, Oven, < 4 Hours","sugar, margarine, egg, flour, salt, buttermilk...",228.0,7.1,1.7,24.5,37.5,0.5,24.7,4.2,"Preheat oven to 350°F., Make pie crust, using ...",80


In [32]:
recepies_df_filterd['category'].value_counts().head(160)
# i think we can decide it doesnt really matter which quisine it is, so we dont take towards the end of the list
# print(recepies_df_filterd['category'].value_counts().count())


category
Dessert           58645
Lunch/Snacks      30956
One Dish Meal     28420
Vegetable         25742
Breakfast         20364
                  ...  
Collard Greens      136
Lime                136
Beginner Cook       135
Bass                135
Pumpkin             134
Name: count, Length: 160, dtype: int64

In [33]:

# Auto-classify all categories into semantic groups using keyword matching
all_categories = recepies_df_filterd['category'].value_counts()

KEYWORD_GROUPS = {
    "meal_type":  ["breakfast", "brunch", "lunch", "dinner", "supper", "dessert",
                   "snack", "appetizer", "side dish", "entree", "main dish",
                   "salad", "soup", "beverage", "drink", "smoothie", "cocktail", "punch"],
    "time":       ["minute", "hour", " min", "quick", "fast", "easy", "30-", "15-", "speed"],
    "dietary":    ["vegan", "vegetarian", "gluten", "low carb", "low fat", "low calorie",
                   "low sodium", "low cholesterol", "low protein", "low sugar",
                   "high protein", "high fiber", "diabetic", "kosher", "halal",
                   "lactose", "dairy free", "nut free", "paleo", "keto", "whole30",
                   "heart healthy", "healthy", "diet", "weight"],
    "cuisine":    ["american", "italian", "mexican", "asian", "chinese", "japanese",
                   "indian", "thai", "french", "greek", "spanish", "mediterranean",
                   "middle east", "african", "european", "caribbean", "swedish",
                   "german", "korean", "vietnamese", "turkish", "lebanese", "moroccan",
                   "canadian", "australian", "brazilian", "peruvian", "cuban",
                   "filipino", "indonesian", "polynesian"],
    "skill":      ["beginner", "advanced", "easy", "technique", "for two",
                   "for one", "college", "student"],
    "occasion":   ["christmas", "thanksgiving", "halloween", "easter", "ramadan",
                   "hanukkah", "passover", "birthday", "wedding", "party",
                   "super bowl", "summer", "winter", "spring", "fall", "holiday",
                   "new year", "valentine", "mother", "father", "4th of july",
                   "memorial day", "labor day"],
    "cooking_method": ["baked", "grilled", "fried", "steamed", "roasted", "broiled",
                       "slow cook", "pressure cook", "stir fry", "microwave",
                       "no bake", "no cook", "one pot", "one pan", "sheet pan",
                       "freezer", "make ahead", "overnight"],
    "ingredient_based": ["chicken", "beef", "pork", "lamb", "turkey", "fish",
                         "shrimp", "seafood", "pasta", "rice", "potato", "mushroom",
                         "tomato", "lemon", "cheese", "egg", "tofu", "beans",
                         "pumpkin", "squash", "spinach", "broccoli", "avocado",
                         "chocolate", "apple", "banana", "berry", "mango"],
}

classified = {group: [] for group in KEYWORD_GROUPS}
classified["other"] = []

for cat in all_categories.index:
    cat_lower = cat.lower()
    matched = False
    for group, keywords in KEYWORD_GROUPS.items():
        if any(kw in cat_lower for kw in keywords):
            classified[group].append(cat)
            matched = True
            break
    if not matched:
        classified["other"].append(cat)

# Print summary
for group, cats in classified.items():
    print(f"\n{'='*40}")
    print(f"  {group.upper()}  ({len(cats)} categories)")
    print(f"{'='*40}")
    for c in cats:
        print(f"  [{all_categories[c]:>5}]  {c}")



  MEAL_TYPE  (12 categories)
  [58645]  Dessert
  [30956]  Lunch/Snacks
  [20364]  Breakfast
  [15238]  Beverages
  [ 4940]  Salad Dressings
  [ 3554]  Frozen Desserts
  [ 3525]  Smoothies
  [ 1956]  Punch Beverage
  [ 1319]  Clear Soup
  [  575]  Brunch
  [    3]  Breakfast Eggs
  [    1]  Desserts Fruit

  TIME  (6 categories)
  [10278]  Quick Breads
  [ 9719]  < 60 Mins
  [ 9020]  < 30 Mins
  [ 6492]  < 15 Mins
  [ 4553]  < 4 Hours
  [    6]  Easy

  DIETARY  (9 categories)
  [ 6214]  Low Protein
  [ 3452]  Low Cholesterol
  [ 3121]  Very Low Carbs
  [ 1598]  Healthy
  [ 1080]  High Protein
  [  785]  Vegan
  [  398]  Lactose Free
  [  132]  Kosher
  [   75]  High Fiber

  CUISINE  (30 categories)
  [ 3453]  European
  [ 2095]  Asian
  [ 1930]  Mexican
  [  527]  Greek
  [  502]  Chinese
  [  419]  Thai
  [  380]  Canadian
  [  364]  Australian
  [  309]  Japanese
  [  308]  Southwest Asia (middle East)
  [  253]  African
  [  249]  Spanish
  [  229]  Indian
  [  227]  German
  [  

In [37]:
# make tag list
recepies_df_filterd['tags'].value_counts().head(100)

tags
Easy                                             11091
< 60 Mins                                         9788
< 15 Mins, Easy                                   9773
< 4 Hours                                         6564
< 30 Mins                                         6214
                                                 ...  
Dessert, Cookie & Brownie, < 30 Mins, Easy         345
Dessert, < 4 Hours, Easy                           345
Vegetable, < 15 Mins, Easy                         343
< 60 Mins, Easy, Inexpensive                       341
Low Protein, Low Cholesterol, < 15 Mins, Easy      340
Name: count, Length: 100, dtype: int64